# Data mining

## EXERCISE: Association analysis from scratch



### Generate frequent itemsets

Let's find all sets of items with a support greater than some threshold.

We define 4 functions for generating frequent itemsets:
* createC1 - Create first candidate itemsets for k=1
* scanD - Identify itemsets that meet the support threshold
* aprioriGen - Generate the next list of candidates
* apriori - Generate all frequent itemsets

See slides for explanation of functions.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def createC1(dataset):
    "Create a list of candidate item sets of size one."
    c1 = []
    for transaction in dataset:
        for item in transaction:
            if not [item] in c1:
                c1.append([item])
    c1.sort()
    #frozenset because it will be a ket of a dictionary.
    return list(map(frozenset, c1))

def scanD(dataset, candidates, min_support):
    "Returns all candidates that meets a minimum support level"
    sscnt = {}
    for tid in dataset:
        for can in candidates:
            if can.issubset(tid):
                sscnt.setdefault(can, 0)
                sscnt[can] += 1

    num_items = float(len(dataset))
    retlist = []
    support_data = {}
    for key in sscnt:
        support = sscnt[key] / num_items
        if support >= min_support:
            retlist.insert(0, key)
            support_data[key] = support
    return retlist, support_data

def aprioriGen(freq_sets, k):
    "Generate the joint transactions from candidate sets"
    retList = []
    lenLk = len(freq_sets)
    for i in range(lenLk):
        for j in range(i + 1, lenLk):
            L1 = list(freq_sets[i])[:k - 2]
            L2 = list(freq_sets[j])[:k - 2]
            L1.sort()
            L2.sort()
            if L1 == L2:
                retList.append(freq_sets[i] | freq_sets[j]) # | is set union
    return retList


def apriori(dataset, min_support=0.5):
    "Generate a list of candidate item sets"
    C1 = createC1(dataset)
    D = list(map(set, dataset))
    L1, support_data = scanD(D, C1, min_support)
    L = [L1]
    k = 2
    while (len(L[k - 2]) > 0):
        Ck = aprioriGen(L[k - 2], k)
        Lk, supK = scanD(D, Ck, min_support)
        support_data.update(supK)
        L.append(Lk)
        k += 1

    return L, support_data

In [17]:
dataset = [
  ['A', 'C', 'D'],
  ['B', 'C', 'E'],
  ['A', 'B', 'C','E'],
  ['B', 'E']
]

MIN_SUPPORT=0.5

# Generate all candidate itemsets
L, support_data = apriori(dataset, min_support=MIN_SUPPORT)
print('Full list of candidate itemsets:\n', L, '\n')
print('Support values for candidate itemsets:\n', support_data, '\n')

Full list of candidate itemsets:
 [[frozenset({'E'}), frozenset({'B'}), frozenset({'C'}), frozenset({'A'})], [frozenset({'B', 'C'}), frozenset({'E', 'C'}), frozenset({'E', 'B'}), frozenset({'A', 'C'})], [frozenset({'E', 'B', 'C'})], []] 

Support values for candidate itemsets:
 {frozenset({'A'}): 0.5, frozenset({'C'}): 0.75, frozenset({'B'}): 0.75, frozenset({'E'}): 0.75, frozenset({'A', 'C'}): 0.5, frozenset({'E', 'B'}): 0.75, frozenset({'E', 'C'}): 0.5, frozenset({'B', 'C'}): 0.5, frozenset({'E', 'B', 'C'}): 0.5} 



### TODO Exploring support thresholds

* Generate frequent itemsets with a support threshold of 0.7
* How many frequent itemsets do we get at 0.7?
* How many do we get at 0.3?
* Do you have datasets that resemble transactions?
* What about the apps/websites you use?

In [18]:
SUPPORT_THRESHOLD_1 = .7
SUPPORT_THRESHOLD_2 = .3

frequent_itemsets = ml.apriori(
  dataset,
  min_support=SUPPORT_THRESHOLD_1,
  use_colnames=True
)

# Step 2: Display results
# print('Frequent Itemsets:')
# print(frequent_itemsets)

AttributeError: 'list' object has no attribute 'size'

## Association analysis using mlxtend library

In [ ]:
#Install the library if it is not available
#!pip install mlxtend
import pandas as pd
import mlxtend.frequent_patterns as ml
from mlxtend.preprocessing import TransactionEncoder

dataset = [['A', 'C', 'D'],
           ['B', 'C', 'E'],
           ['A', 'B', 'C','E'],
           ['B', 'E']]

oht = TransactionEncoder()
oht_ary = oht.fit(dataset).transform(dataset)
df = pd.DataFrame(oht_ary, columns=oht.columns_)
# Convert True/False → 1/0
#df = df.astype(int)
print (df)

# Step 1: Get frequent itemsets using Apriori
frequent_itemsets = ml.apriori(df, min_support=0.5, use_colnames=True)

# Step 2: Display results
print("Frequent Itemsets:")
print(frequent_itemsets)

# Step 3:
print('\n----------------\nLeave space')
print(frequent_itemsets)

       A      B      C      D      E
0   True  False   True   True  False
1  False   True   True  False   True
2   True   True   True  False   True
3  False   True  False  False   True
Frequent Itemsets:
   support   itemsets
0     0.50        (A)
1     0.75        (B)
2     0.75        (C)
3     0.75        (E)
4     0.50     (A, C)
5     0.50     (B, C)
6     0.75     (E, B)
7     0.50     (E, C)
8     0.50  (E, B, C)

----------------
Leave space
   support   itemsets
0     0.50        (A)
1     0.75        (B)
2     0.75        (C)
3     0.75        (E)
4     0.50     (A, C)
5     0.50     (B, C)
6     0.75     (E, B)
7     0.50     (E, C)
8     0.50  (E, B, C)


### TODO Exploring support thresholds

* Generate frequent itemsets with a support threshold of 0.7
* How many frequent itemsets do we get at 0.7?
* How many do we get at 0.3?


In [ ]:
# TODO: replace the content of this cell with your Python solution


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## *STOP PLEASE. THE FOLLOWING IS FOR THE NEXT EXERCISE. THANKS.*

## EXERCISE:  Mine association rules

Given frequent itemsets, we can create association rules.

We add three more functions:
* calc_confidence - Identify rules that meet the confidence threshold
* rules_from_conseq - Recursively generate and evaluate candidate rules
* generateRules - Mine all confident association rules



In [ ]:
def calc_confidence(freqSet, H, support_data, rules, min_confidence=0.7):
    "Evaluate the rule generated"
    pruned_H = []
    for conseq in H:
        conf = support_data[freqSet] / support_data[freqSet - conseq]
        if conf >= min_confidence:
            #print(freqSet - conseq, '--->', conseq, 'conf:', conf)
            rules.append((freqSet - conseq, conseq, conf))
            pruned_H.append(conseq)
    return pruned_H


def rules_from_conseq(freqSet, H, support_data, rules, min_confidence=0.7):
    "Generate a set of candidate rules"
    m = len(H[0])
    Hmp1 = createC1(H)
    Hmp1 = calc_confidence(freqSet, Hmp1,  support_data, rules, min_confidence)
    if len(Hmp1) <= len(freqSet):
        if (len(freqSet) > (m + 1)):
            Hmp1 = aprioriGen(H, m + 1)
            Hmp1 = calc_confidence(freqSet, Hmp1,  support_data, rules, min_confidence)
            if len(Hmp1) > 1:
                rules_from_conseq(freqSet, Hmp1, support_data, rules, min_confidence)

def generateRules(L, support_data, min_confidence=0.7):
    """Create the association rules
    L: list of frequent item sets
    support_data: support data for those itemsets
    min_confidence: minimum confidence threshold
    """
    rules = []
    for i in range(1, len(L)):
        for freqSet in L[i]:
            H1 = [frozenset([item]) for item in freqSet]
            if (i > 1):
                rules_from_conseq(freqSet, H1, support_data, rules, min_confidence)
            else:
                calc_confidence(freqSet, H1, support_data, rules, min_confidence)
    return rules

def print_rules(rules):
    for r in rules:
        print('{} ==> {} (c={})'.format(*r))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
dataset = [['A', 'C', 'D'],
           ['B', 'C', 'E'],
           ['A', 'B', 'C','E'],
           ['B', 'E']]
L, support_data = apriori(dataset, min_support=0.5)
MIN_CONFIDENCE = 0.3
# Mine association rules
association_rules = generateRules(L, support_data, min_confidence=MIN_CONFIDENCE)
print_rules(association_rules)

frozenset({'C'}) ==> frozenset({'B'}) (c=0.6666666666666666)
frozenset({'B'}) ==> frozenset({'C'}) (c=0.6666666666666666)
frozenset({'C'}) ==> frozenset({'E'}) (c=0.6666666666666666)
frozenset({'E'}) ==> frozenset({'C'}) (c=0.6666666666666666)
frozenset({'B'}) ==> frozenset({'E'}) (c=1.0)
frozenset({'E'}) ==> frozenset({'B'}) (c=1.0)
frozenset({'C'}) ==> frozenset({'A'}) (c=0.6666666666666666)
frozenset({'A'}) ==> frozenset({'C'}) (c=1.0)
frozenset({'E', 'C'}) ==> frozenset({'B'}) (c=1.0)
frozenset({'E', 'B'}) ==> frozenset({'C'}) (c=0.6666666666666666)
frozenset({'B', 'C'}) ==> frozenset({'E'}) (c=1.0)
frozenset({'C'}) ==> frozenset({'E', 'B'}) (c=0.6666666666666666)
frozenset({'B'}) ==> frozenset({'E', 'C'}) (c=0.6666666666666666)
frozenset({'E'}) ==> frozenset({'B', 'C'}) (c=0.6666666666666666)
frozenset({'E', 'C'}) ==> frozenset({'B'}) (c=1.0)
frozenset({'E', 'B'}) ==> frozenset({'C'}) (c=0.6666666666666666)
frozenset({'B', 'C'}) ==> frozenset({'E'}) (c=1.0)


## Association analysis using mlxtend library

In [ ]:
#Install the library if it is not available
#!pip install mlxtend
import pandas as pd
import mlxtend.frequent_patterns as ml
from mlxtend.preprocessing import TransactionEncoder

dataset = [['A', 'C', 'D'],
           ['B', 'C', 'E'],
           ['A', 'B', 'C','E'],
           ['B', 'E']]

oht = TransactionEncoder()
oht_ary = oht.fit(dataset).transform(dataset)
df = pd.DataFrame(oht_ary, columns=oht.columns_)
# Convert True/False → 1/0
df = df.astype(int)
print (df)

# Step 1: Get frequent itemsets using Apriori
frequent_itemsets = ml.apriori(df, min_support=0.7, use_colnames=True)

# Step 2: Generate association rules
rules= ml.association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

# Step 3: Display results
print("Frequent Itemsets:")
print(frequent_itemsets)
print("\nAssociation Rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

   A  B  C  D  E
0  1  0  1  1  0
1  0  1  1  0  1
2  1  1  1  0  1
3  0  1  0  0  1
Frequent Itemsets:
   support itemsets
0     0.75      (B)
1     0.75      (C)
2     0.75      (E)
3     0.75   (E, B)

Association Rules:
  antecedents consequents  support  confidence      lift
0         (E)         (B)     0.75         1.0  1.333333
1         (B)         (E)     0.75         1.0  1.333333


/usr/local/lib/python3.11/dist-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


### TODO Exploring confidence thresholds

* Mine rules with a confidence threshold of 0.7
* How many rules do we get at 0.7?
* How many do we get at 0.3?
* Can we use this for recommendation (e.g., Amazon, Netflix)?

In [ ]:
# TODO: replace the content of this cell with your Python solution


# Load the  supermarket transaction datasets
### Now Lets work on a real Grocery dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import csv
import pprint

file_path = "/content/drive/MyDrive/ColabNotebooks/36100/wk04_association_rules/Groceries.csv"
#file_path = 'Groceries.csv'
# Check if the file exists
import os
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")
data_list = []
with open(file_path, 'r') as f:  #opens PW file
    reader = csv.reader(f)
    # Print every value of every row.
    for row in reader:
        row_list = []
        for value in row:
            if len(value.strip()) > 0 and value.strip() != '':
                row_list.append(value.strip())
        data_list.append(row_list)
#pprint.pprint(data_list)

## TODO Mining frequent items on Groceries datasets
* Apply apriori functions from mlxtend library
* What would be a reasonable value of min-support for these supermarket transaction data

In [ ]:
# TODO: replace the content of this cell with your Python solution
raise NotImplementedError

## TODO Mining association rules on Groceries datasets
* Apply association_rules functions from mlxtend library


In [ ]:
# TODO: replace the content of this cell with your Python solution
raise NotImplementedError

## *STOP PLEASE. THE FOLLOWING IS FOR THE NEXT EXERCISE. THANKS.*

# EXERCISE: FP-Growth
## Rules  generation using pyfpgrowth library


In [ ]:
#Install the library if it is not available
#!pip install pyfpgrowth
import pyfpgrowth
MIN_SUPPORT = 2
MIN_CONFIDENCE = 0.7
DATASET = [['A', 'C', 'D'], ['B', 'C', 'E'], ['A', 'B', 'C','E'],['B', 'E']]

frequent_itemsets = pyfpgrowth.find_frequent_patterns(DATASET, MIN_SUPPORT)
print('Support values for candidate itemsets:\n', frequent_itemsets, '\n')
rules = pyfpgrowth.generate_association_rules(frequent_itemsets, MIN_CONFIDENCE)
print('Resultant assoication rules:\n')
pprint.pprint(rules)

ModuleNotFoundError: No module named 'pyfpgrowth'

## Rules  generation using fpgrowth_py library


In [ ]:
#!pip install fpgrowth_py
from fpgrowth_py import fpgrowth
freqItemSet, rules = fpgrowth(DATASET, minSupRatio=0.5, minConf=0.7)
for r in rules:
    print('{} ==> {} (c={})'.format(*r))

ModuleNotFoundError: No module named 'fpgrowth_py'

### TODO Mining association rules using FP-growth  on Groceries datasets
* Try different confidence thresholds
* What’s a reasonable value for real data?

In [ ]:
# TODO: replace the content of this cell with your Python solution
raise NotImplementedError

## *STOP PLEASE. THE FOLLOWING IS FOR THE NEXT EXERCISE. THANKS.*

# Recommender System

# Here’s a progressive Python exercise plan for practicing Matrix Factorization (MF) for recommender systems, starting small and moving to real data.

## Step 1: Small toy dataset (practice)

## Objective: Implement MF with gradient descent on a tiny user-item matrix.

In [ ]:
import numpy as np

# Step 1: Small user-item rating matrix
# Rows = users, Columns = items
R = np.array([
    [5, 3, 0],
    [4, 0, 0],
    [1, 1, 0],
    [0, 0, 5],
    [0, 0, 4]
])

# Hyperparameters
k = 2          # number of latent features
steps = 50     # number of iterations
alpha = 0.01   # learning rate (for GD)
lambda_ = 0.1  # regularization

# Initialize user (U) and item (V) matrices randomly
num_users, num_items = R.shape
U = np.random.rand(num_users, k)
V = np.random.rand(num_items, k)

# Optional: Implement simple gradient descent update (MF)
for step in range(steps):
    for i in range(num_users):
        for j in range(num_items):
            if R[i,j] > 0:  # only consider observed ratings
                eij = R[i,j] - U[i,:].dot(V[j,:].T)
                U[i,:] += alpha * (eij * V[j,:] - lambda_ * U[i,:])
                V[j,:] += alpha * (eij * U[i,:] - lambda_ * V[j,:])

# Predicted full rating matrix
R_pred = U.dot(V.T)
print("Predicted Ratings:\n", R_pred)


Predicted Ratings:
 [[4.80384489 2.84053041 5.78760828]
 [3.75195463 2.10413035 4.56399705]
 [1.31642835 0.83165012 1.56568357]
 [3.9272142  2.27005131 4.75136047]
 [3.22133364 1.7241641  3.94999544]]


## Objective: Implement MF with ALS on a tiny user-item matrix.

In [ ]:
from sklearn.decomposition import NMF
import numpy as np

R = np.array([
    [5, 3, 0],
    [4, 0, 0],
    [1, 1, 0],
    [0, 0, 5],
    [1, 0, 4]
])

model = NMF(n_components=2, init='random', random_state=42, max_iter=5000)
U = model.fit_transform(R)
V = model.components_
pred_ratings = np.dot(U, V)

print("Predicted Ratings:\n", pred_ratings)


Predicted Ratings:
 [[5.3373265  2.17059571 0.        ]
 [3.43236352 1.395778   0.01201179]
 [1.20705417 0.49088745 0.        ]
 [0.10554109 0.         4.9977624 ]
 [0.86993092 0.31940902 4.00275778]]


# Tasks:
*  Modify k (latent features) and see the effect on predictions.

*  Compare results using ALS instead of gradient descent.





In [ ]:
# TODO: replace the content of this cell with your Python solution
raise NotImplementedError

# Real dataset (practice)

Dataset suggestion: MovieLens 100K

Objective: Implement MF on real-world ratings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/content/drive/MyDrive/ColabNotebooks/36100/wk04_association_rules/"

In [ ]:

import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# ---------------------------
# 1. Load the dataset
# ---------------------------
columns = ['user_id', 'item_id', 'rating', 'timestamp']
data = pd.read_csv(file_path+"ml-100k/u.data", sep='\t', names=columns)

# Drop timestamp since not needed
data = data.drop("timestamp", axis=1)

# ---------------------------
# 2. Train-Test Split
# ---------------------------
train, test = train_test_split(data, test_size=0.2, random_state=42)

# ---------------------------
# 3. Create User-Item Matrices
# ---------------------------
# Full set of users and items
users = data["user_id"].unique()
items = data["item_id"].unique()

# Re-index users and items (matrix indices must be continuous)
user_index = {u: i for i, u in enumerate(users)}
item_index = {i: j for j, i in enumerate(items)}

def build_matrix(df):
    mat = np.zeros((len(users), len(items)))
    for row in df.itertuples():
        mat[user_index[row.user_id], item_index[row.item_id]] = row.rating
    return mat

R_train = build_matrix(train)
R_test = build_matrix(test)

# ---------------------------
# 4. Apply NMF on Train
# ---------------------------
nmf_model = NMF(n_components=20, init="random", random_state=42, max_iter=300)
U = nmf_model.fit_transform(R_train)
V = nmf_model.components_
R_pred = np.dot(U, V)

# ---------------------------
# 5. Evaluate on Test
# ---------------------------
# Collect test ratings and predicted ratings
y_true, y_pred = [], []
for row in test.itertuples():
    u = user_index[row.user_id]
    i = item_index[row.item_id]
    y_true.append(row.rating)
    y_pred.append(R_pred[u, i])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"Test RMSE: {rmse:.4f}")


Test RMSE: 2.6180


/usr/local/lib/python3.11/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 300 reached. Increase it to improve convergence.
  warnings.warn(


# Generate Top-N movie recommendations for a given user.

In [ ]:
# ---------------------------
# 6. Top-N Recommendations for a User
# ---------------------------
# Load movie titles for readability
movie_titles = pd.read_csv(file_path+"ml-100k/u.item", sep="|", encoding="latin-1", header=None)
movie_titles = movie_titles[[0, 1]]
movie_titles.columns = ["item_id", "title"]

def recommend_movies(user_id, N=10):
    # Get index of the user in the matrix
    u = user_index[user_id]

    # Get predicted ratings for this user
    preds = R_pred[u]

    # Get the items the user has already rated
    rated_items = set(data[data.user_id == user_id]["item_id"])

    # Rank all items by predicted rating
    item_ranking = np.argsort(preds)[::-1]  # descending

    recommendations = []
    for idx in item_ranking:
        item_id = items[idx]
        if item_id not in rated_items:  # exclude already rated
            title = movie_titles[movie_titles.item_id == item_id]["title"].values[0]
            score = preds[idx]
            recommendations.append((title, score))
        if len(recommendations) >= N:
            break

    return recommendations

# Example: Recommend 10 movies for user 50
user_id = 50
top_movies = recommend_movies(user_id, N=10)

print(f"\nTop 10 Recommendations for User {user_id}:\n")
for rank, (title, score) in enumerate(top_movies, 1):
    print(f"{rank}. {title} (predicted rating: {score:.2f})")



Top 10 Recommendations for User 50:

1. Godfather, The (1972) (predicted rating: 1.20)
2. Bound (1996) (predicted rating: 1.11)
3. Big Night (1996) (predicted rating: 0.95)
4. Mighty Aphrodite (1995) (predicted rating: 0.92)
5. Secrets & Lies (1996) (predicted rating: 0.91)
6. Donnie Brasco (1997) (predicted rating: 0.89)
7. Swingers (1996) (predicted rating: 0.85)
8. Grosse Pointe Blank (1997) (predicted rating: 0.84)
9. Welcome to the Dollhouse (1995) (predicted rating: 0.80)
10. Heat (1995) (predicted rating: 0.79)


# TODO
* Apply NMF with k = 5, 10, 20 latent factors.

* Compute RMSE for each case.

* Plot RMSE vs k.
* Select a few users and items where ratings are missing in the original matrix.

* Predict those ratings using the trained NMF model.

* Compare predictions with actual ratings if available.





In [ ]:
# TODO: replace the content of this cell with your Python solution



# End of Tutorial. Many Thanks.